# Video Action Recognition
Tian Liu

This notebook prepares the UCF50 dataset (frame extraction and a group-held-out train/validation/test split), runs training and evaluation through the bash scripts in `scripts/`, and fuses an RGB stream with an RGB-difference (motion) stream by averaging their predicted class probabilities. The fused two-stream LRCN reaches 93.88% test accuracy, compared with 89.43% for RGB alone.

## Setup

In [1]:
from google.colab import drive
import torch
import os
import glob
import numpy as np
import cv2
from sklearn.metrics import classification_report

In [2]:
#Mount Google Drive and navigate to the assignment directory
drive.mount('/content/drive')
%cd /content/drive/MyDrive/assignments/Video_action_recognition
!git pull
%pip install -q -r requirements.txt
%env PYTHONPATH=src
%env UCF50_N_FRAMES=16
with open("/content/drive/MyDrive/credentials/wandb_api.txt") as f:
    os.environ["WANDB_API_KEY"] = f.read().strip()
print("W&B key loaded:", len(os.environ.get("WANDB_API_KEY", "")) > 0)
# Check for PyTorch and CUDA availability
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/assignments/Video_action_recognition
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 3.08 KiB | 2.00 KiB/s, done.
From https://github.com/ttl074/Video_action_recognition
   d9b8bc7..7e04ae7  main       -> origin/main
Updating d9b8bc7..7e04ae7
Fast-forward
 scripts/run.py        | 52 ++++++++++++++++++++++++++++++---------
 src/models.py         | 68 +++++++++++++++++++++++++++++++++++++++++----------
 src/video_datasets.py | 51 +++++++++++++++++++++++++-------------
 3 files changed, 129 insertions(+), 42 deletions(-)
env: PYTHONPATH=src
env: UCF50_N_FRAMES=16
W&B key loaded: True
PyTorch version: 2.11.0+cu130
CUDA available: True


In [3]:
# Setup paths for dataset
drive_path = '/content/drive/MyDrive/datasets/ucf50'
local_path = '/content'
url = 'https://www.crcv.ucf.edu/data/UCF50/UCF50.rar'
extract_path = os.path.join(local_path, 'datasets')
raw_dir = os.path.join(extract_path, 'UCF50')
frame_dir = os.path.join(extract_path,'UCF50_frames')

In [4]:
!python scripts/prep_dataset.py --drive_path {drive_path} --local_path {local_path} --url {url}

Local rar file already exists. Skipping download.
Extract path already exists at /content/datasets/UCF50. Skipping extraction.


In [5]:
video_counts = {}
frame_counts = []
failed_videos = []
for category in os.listdir('/content/datasets/UCF50'):
    category_path = os.path.join('/content/datasets/UCF50', category)
    video_count = 0
    for video in os.listdir(category_path):
        video_path = os.path.join(category_path, video)
        v_cap = cv2.VideoCapture(video_path)
        if not v_cap.isOpened():
            print("Failed to open video:", video_path)
            failed_videos.append(video_path)
            v_cap.release()
            continue
        v_len = int(v_cap.get(cv2.CAP_PROP_FRAME_COUNT))
        v_cap.release()
        frame_counts.append(v_len)
        video_count += 1
    video_counts[category] = video_count
largest_category = max(video_counts, key=video_counts.get)
smallest_category = min(video_counts, key=video_counts.get)
print("Number of videos:", sum(video_counts.values()))
print("Largest category:", largest_category, video_counts[largest_category])
print("Smallest category:", smallest_category, video_counts[smallest_category])
print("Median frame count:", np.median(frame_counts))
print("Min frame count:", np.min(frame_counts))
print("Max frame count:", np.max(frame_counts))
print("Failed videos:", failed_videos)

Number of videos: 6681
Largest category: HorseRiding 197
Smallest category: PlayingViolin 100
Median frame count: 181.0
Min frame count: 29
Max frame count: 900
Failed videos: []


In [6]:
n_frames = 16
!python scripts/extract_frames.py --raw_dir {raw_dir} --n_frames {n_frames} --frame_dir {frame_dir}

Converting from avi: 100% 6681/6681 [00:00<00:00, 56477.36file/s]
Extracted 0 and skipped 6681 out of 6681 avis


## Tests

In [7]:
!python -m pytest -v -rs tests/

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/drive/MyDrive/assignments/Video_action_recognition
configfile: pytest.ini
plugins: langsmith-0.14.11, anyio-4.15.1, typeguard-4.6.0, platformdirs-4.13.5
collected 13 items                                                             

tests/test_dataset.py::test_raw_class_folders PASSED                     [  7%]
tests/test_dataset.py::test_video_avis PASSED                            [ 15%]
tests/test_dataset.py::test_ucf50_frames_dir PASSED                      [ 23%]
tests/test_dataset.py::test_video_dirs PASSED                            [ 30%]
tests/test_dataset.py::test_frame_counts PASSED                          [ 38%]
tests/test_dataset.py::test_frames_read PASSED                           [ 46%]
tests/test_models.py::test_batch_independence PASSED                     [ 53%]
te

## Training

In [8]:
!bash scripts/train.sh --cnn_backbone resnet50 --dropout 0.3 --num_workers 7 --amp --fr_per_vid 16 --diff_stack 3 --rnn_hidden_size 512 --input_mode diff --run_name r50v2_diff3_fpv16_h512

Loading video dataset....
100% 50/50 [00:00<00:00, 6588.19it/s]
Splitting train/validation/test datasets....
Dropping 63 videos in groups outside 1-25
Train groups: [ 1  2  3  4  7  9 10 12 13 14 15 16 17 18 19 21 22 23 24]
Validation groups: [ 6 20]
Test groups: [ 5  8 11 25]
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttl074 (ttl074-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run dvaab23j (0.2s)
wandb: ⣯ setting up run dvaab23j (0.2s)
wandb: ⣟ setting up run dvaab23j (0.2s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/assignments/Video_action_recognition/wandb/run-20261027_033221-dvaab23j
wandb: Run `wandb offline` to turn off syncing.


## Testing

In [9]:
!bash scripts/test.sh models/r50v2_diff3_fpv16_h512/best_model_wts.pt --cnn_backbone resnet50 --rnn_hidden_size 512 --fr_per_vid 16 --input_mode diff --diff_stack 3 --split val --save_probs probs/val_diff3.npz --run_name val_diff3_probs
!bash scripts/test.sh models/r50v2_diff3_fpv16_h512/best_model_wts.pt --cnn_backbone resnet50 --rnn_hidden_size 512 --fr_per_vid 16 --input_mode diff --diff_stack 3 --save_probs probs/diff3.npz --run_name test_diff3_probs

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: ttl074 (ttl074-johns-hopkins-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run g4o990x9 (0.3s)
wandb: ⣯ setting up run g4o990x9 (0.3s)
wandb: ⣟ setting up run g4o990x9 (0.3s)
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /content/drive/MyDrive/assignments/Video_action_recognition/wandb/run-20261027_090426-g4o990x9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run val_diff3_probs
wandb: ⭐️ View project at https://wandb.ai/ttl074-johns-hopkins-university/Video_Action_Recognition
wandb: 🚀 View run at https://wandb.ai/ttl074-johns-hopkins-university/Video_Action_Recognition/runs/g4o990x9
100% 68/68 [01:17<00:00,  1.15s/it

## Stream Fusion

In [10]:
def load(path):
    d = np.load(path)
    return d["probs"], d["labels"]

# Validation and test probabilities for each stream
streams = {
    "rgb":   ("probs/val_run6_rgb.npz", "probs/run6_rgb.npz"),
    "diff":  ("probs/val_diff.npz",     "probs/diff.npz"),
    "diff3": ("probs/val_diff3.npz",    "probs/diff3.npz"),
}
val = {k: load(v[0]) for k, v in streams.items()}
test = {k: load(v[1]) for k, v in streams.items()}
val_y, test_y = val["rgb"][1], test["rgb"][1]
assert all(np.array_equal(val_y, v[1]) for v in val.values()), "val order differs"
assert all(np.array_equal(test_y, t[1]) for t in test.values()), "test order differs"

In [11]:
def acc(probs, y):
    return (probs.argmax(1) == y).mean()

# 1. Choose the motion stream and the RGB weight on VALIDATION only
results = []
for motion in ("diff", "diff3"):
    for w in (0.3, 0.4, 0.5, 0.6, 0.7):
        a = acc(w * val["rgb"][0] + (1 - w) * val[motion][0], val_y)
        results.append((a, -abs(w - 0.5), motion, w))  # ties: prefer w closest to 0.5
        print(f"rgb + {motion:5s} w_rgb={w}: val {100 * a:.2f}%")
best_val, _, motion, w = max(results)
print(f"\nChosen on validation: rgb + {motion}, w_rgb={w} (val {100 * best_val:.2f}%)")

# 2. Apply that one choice to TEST, once
fused = w * test["rgb"][0] + (1 - w) * test[motion][0]
print(f"\nSingle streams on test: rgb {100 * acc(test['rgb'][0], test_y):.2f}%, "
      f"{motion} {100 * acc(test[motion][0], test_y):.2f}%")
print(f"Fused on test: {100 * acc(fused, test_y):.2f}%")

classes = sorted(os.listdir("/content/datasets/UCF50_frames"))
print(classification_report(test_y, fused.argmax(1), labels=range(len(classes)),
                            target_names=classes, digits=3))

rgb + diff  w_rgb=0.3: val 91.34%
rgb + diff  w_rgb=0.4: val 91.90%
rgb + diff  w_rgb=0.5: val 93.92%
rgb + diff  w_rgb=0.6: val 93.55%
rgb + diff  w_rgb=0.7: val 93.37%
rgb + diff3 w_rgb=0.3: val 90.98%
rgb + diff3 w_rgb=0.4: val 93.00%
rgb + diff3 w_rgb=0.5: val 95.21%
rgb + diff3 w_rgb=0.6: val 93.74%
rgb + diff3 w_rgb=0.7: val 93.19%

Chosen on validation: rgb + diff3, w_rgb=0.5 (val 95.21%)

Single streams on test: rgb 89.43%, diff3 91.38%
Fused on test: 93.88%
                    precision    recall  f1-score   support

     BaseballPitch      1.000     1.000     1.000        26
        Basketball      1.000     0.905     0.950        21
        BenchPress      1.000     1.000     1.000        26
            Biking      1.000     1.000     1.000        25
         Billiards      1.000     1.000     1.000        26
      BreastStroke      1.000     1.000     1.000        16
      CleanAndJerk      0.889     1.000     0.941        16
            Diving      0.957     1.000     0.97

## Conclusion

### Results (test set, group-held-out: train, validation, and test share no UCF50 groups)

| Model | Input | Test accuracy |
|---|---|---|
| LRCN, ResNet34 (run 3) | RGB, 16 frames | 86.65% |
| LRCN, ResNet50 (run 5) | RGB, 16 frames | 89.25% (89.71% with 3-clip testing) |
| LRCN, ResNet50, LSTM 512 (run 6) | RGB, 32 frames | 89.43% |
| LRCN, ResNet50, LSTM 512 | RGB difference, 32 positions × 1 | 89.34% |
| LRCN, ResNet50, LSTM 512 | RGB difference, 16 positions × 3 stacked | 91.38% |
| **Two-stream fusion** | **RGB (run 6) + stacked differences** | **93.88%** |

The fusion weight and the choice of motion stream were selected on the validation split (equal weights scored best); the test set was evaluated once with that choice.

### Findings

- **RGB-only LRCN plateaued around 89.5%.** Changing the backbone, frame count, and LSTM size moved test accuracy by less than one point, while per-class errors shifted between classes. The remaining errors were concentrated in fast, repetitive actions (JumpingJack, Nunchucks, Lunges, JugglingBalls) that look alike in single frames.
- **Motion input addressed those classes directly.** A model trained only on RGB differences matched the RGB model overall but failed on different classes: it improved JumpingJack (33% → 62%), Nunchucks (52% → 74%), and JugglingBalls (75% → 100%), and was weaker on scene-defined classes such as Biking, Drumming, and MilitaryParade.
- **Stacking consecutive differences improved the motion stream by 2 points** at equal data-loading cost, by giving each time step a short motion window instead of a single instant.
- **Fusing the two streams gave the largest gain** (+4.5 points over RGB alone), because their errors barely overlap: the stream suited to each class tends to be the more confident one.

### Limitations

- Each configuration was trained once with a single seed, and the validation split is only 2 groups, so differences under about one point are within noise.
- The stacked-difference run changed two settings at once (positions 32 → 16, stack 1 → 3), so its gain cannot be attributed to stacking alone.
- The two-stream model needs two forward passes per video, twice the inference cost of a single model.
- 63 clips in groups 26–30 were excluded so that every class has the same 25 groups.

### Future work

- Replace RGB differences with optical flow, the motion input used by the original two-stream networks.
- Regularize the RGB stream (label smoothing, temporal mean-pooling of the LSTM outputs), which reaches about 99.7% training accuracy.
- Initialize from video-pretrained (Kinetics-400) weights, if permitted.